In [ ]:
!pip -q install numpy scipy scikit-learn h5py onnx onnxscript onnxruntime skl2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 12.7 MB/s eta 0:00:00


In [ ]:
import os
import gc
import glob
import json
import time
import random

import h5py
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
import onnxruntime as ort

from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "/content/drive/MyDrive/global_neurotech_hack_2026/medium_data"
DATASET_KEY = "acquisition/ElectricalSeries"

TOTAL_CHANNELS = 76
ENCODED_CHANNELS = 64
GT_CHANNELS = 12

FS = 32000
BIN_SIZE = 320          # 10 ms
CONTEXT_BINS = 20       # 200 ms context
CHUNK_BINS = 256

CACHE_DIR = "/content/binned_cache"
OUT_DIR = "/content/decoder_outputs_ridge"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.h5")))
assert len(FILES) > 0, f"No .h5 files found in {DATA_DIR}"

print("Found files:")
for f in FILES:
    print(" -", os.path.basename(f))

Mounted at /content/drive
Found files:
 - smoke_Ani_seed1.h5
 - smoke_Ani_seed2.h5
 - smoke_Ani_seed3.h5
 - smoke_Ani_seed4.h5
 - smoke_Ani_seed5.h5


In [ ]:
def bin_file_to_arrays(
    h5_path: str,
    dataset_key: str = DATASET_KEY,
    total_channels: int = TOTAL_CHANNELS,
    encoded_channels: int = ENCODED_CHANNELS,
    bin_size: int = BIN_SIZE,
    chunk_bins: int = CHUNK_BINS,
    cache_dir: str = CACHE_DIR,
):
    """
    Reads the flattened ElectricalSeries dataset and bins by averaging every bin_size raw frames.
    Returns:
        Xb: (n_bins, 64) float32
        Yb: (n_bins, 12) float32
    """
    stem = os.path.splitext(os.path.basename(h5_path))[0]
    cache_path = os.path.join(cache_dir, f"{stem}_bin{bin_size}.npz")

    if os.path.exists(cache_path):
        z = np.load(cache_path)
        Xb = z["X"].astype(np.float32)
        Yb = z["Y"].astype(np.float32)
        print(f"[cache] {stem}: X={Xb.shape}, Y={Yb.shape}")
        return Xb, Yb

    X_parts = []
    Y_parts = []

    with h5py.File(h5_path, "r") as f:
        dset = f[dataset_key]
        flat_len = int(dset.shape[0])

        assert flat_len % total_channels == 0, (
            f"{h5_path}: flat length {flat_len} not divisible by {total_channels}"
        )

        n_frames = flat_len // total_channels
        usable_frames = (n_frames // bin_size) * bin_size
        usable_values = usable_frames * total_channels

        values_per_chunk = chunk_bins * bin_size * total_channels

        print(f"[read ] {stem}: n_frames={n_frames}, usable_bins={usable_frames // bin_size}")

        for start in range(0, usable_values, values_per_chunk):
            end = min(start + values_per_chunk, usable_values)

            flat = np.asarray(dset[start:end], dtype=np.float32)
            frames = flat.reshape(-1, total_channels)
            binned = frames.reshape(-1, bin_size, total_channels).mean(axis=1)

            X_parts.append(binned[:, :encoded_channels].astype(np.float32))
            Y_parts.append(binned[:, encoded_channels:].astype(np.float32))

            del flat, frames, binned
            gc.collect()

    Xb = np.concatenate(X_parts, axis=0).astype(np.float32)
    Yb = np.concatenate(Y_parts, axis=0).astype(np.float32)

    np.savez(cache_path, X=Xb, Y=Yb)
    print(f"[saved] {stem}: X={Xb.shape}, Y={Yb.shape}")

    return Xb, Yb


binned_files = []
for path in FILES:
    Xb, Yb = bin_file_to_arrays(path)
    binned_files.append((path, Xb, Yb))

print("\nSummary:")
for path, Xb, Yb in binned_files:
    print(os.path.basename(path), "X:", Xb.shape, "Y:", Yb.shape)

[read ] smoke_Ani_seed1: n_frames=6701000, usable_bins=20940
[saved] smoke_Ani_seed1: X=(20940, 64), Y=(20940, 12)
[read ] smoke_Ani_seed2: n_frames=4417184, usable_bins=13803
[saved] smoke_Ani_seed2: X=(13803, 64), Y=(13803, 12)
[read ] smoke_Ani_seed3: n_frames=4345248, usable_bins=13578
[saved] smoke_Ani_seed3: X=(13578, 64), Y=(13578, 12)
[read ] smoke_Ani_seed4: n_frames=4416704, usable_bins=13802
[saved] smoke_Ani_seed4: X=(13802, 64), Y=(13802, 12)
[read ] smoke_Ani_seed5: n_frames=4292416, usable_bins=13413
[saved] smoke_Ani_seed5: X=(13413, 64), Y=(13413, 12)

Summary:
smoke_Ani_seed1.h5 X: (20940, 64) Y: (20940, 12)
smoke_Ani_seed2.h5 X: (13803, 64) Y: (13803, 12)
smoke_Ani_seed3.h5 X: (13578, 64) Y: (13578, 12)
smoke_Ani_seed4.h5 X: (13802, 64) Y: (13802, 12)
smoke_Ani_seed5.h5 X: (13413, 64) Y: (13413, 12)


In [ ]:
def make_windows(Xb: np.ndarray, Yb: np.ndarray, context_bins: int):
    """
    Xb: (T, 64)
    Yb: (T, 12)

    Returns:
        Xw: (N, context_bins*64)
        Yw: (N, 10) for gt_0..gt_9, scaled to controller-like ranges
    """
    T, C = Xb.shape
    N = T - context_bins + 1
    if N <= 0:
        raise ValueError(f"Not enough bins: T={T}, context_bins={context_bins}")

    shape = (N, context_bins, C)
    strides = (Xb.strides[0], Xb.strides[0], Xb.strides[1])

    Xw = np.lib.stride_tricks.as_strided(
        Xb, shape=shape, strides=strides
    ).copy()

    Xw = Xw.reshape(N, context_bins * C).astype(np.float32)

    # Predict only gt_0..gt_9
    Y = Yb[context_bins - 1:, :10].astype(np.float32)

    # Scale outputs to controller-like ranges:
    # gt_0..gt_3 are signed -> [-1, 1]
    # gt_4..gt_9 are nonnegative -> [0, 1]
    Y_scaled = np.empty_like(Y, dtype=np.float32)
    Y_scaled[:, :4] = np.clip(Y[:, :4] / 32767.0, -1.0, 1.0)
    Y_scaled[:, 4:10] = np.clip(Y[:, 4:10] / 32767.0, 0.0, 1.0)

    return Xw, Y_scaled


def make_sample_weights(Y_scaled: np.ndarray):
    """
    Emphasize active bins so the model doesn't collapse to near-zero predictions.
    """
    w = np.ones(len(Y_scaled), dtype=np.float32)

    analog_active = np.max(np.abs(Y_scaled[:, :4]), axis=1) > 0.10
    sparse_active = np.max(Y_scaled[:, 4:10], axis=1) > 0.05

    w += 1.0 * analog_active.astype(np.float32)
    w += 2.0 * sparse_active.astype(np.float32)
    return w

In [ ]:
# Hold out the last file as validation for alpha selection
train_indices = list(range(len(binned_files) - 1))
val_indices = [len(binned_files) - 1]

print("Train files:")
for i in train_indices:
    print(" -", os.path.basename(binned_files[i][0]))

print("Val file:")
for i in val_indices:
    print(" -", os.path.basename(binned_files[i][0]))


def build_split(indices):
    X_list, Y_list, W_list = [], [], []

    for i in indices:
        _, Xb, Yb = binned_files[i]
        Xw, Yw = make_windows(Xb, Yb, CONTEXT_BINS)
        W = make_sample_weights(Yw)

        X_list.append(Xw.astype(np.float32))
        Y_list.append(Yw.astype(np.float32))
        W_list.append(W.astype(np.float32))

        del Xw, Yw, W
        gc.collect()

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    Y = np.concatenate(Y_list, axis=0).astype(np.float32)
    W = np.concatenate(W_list, axis=0).astype(np.float32)
    return X, Y, W


X_train, Y_train, W_train = build_split(train_indices)
X_val, Y_val, W_val = build_split(val_indices)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("W_train:", W_train.shape)

print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)
print("W_val:", W_val.shape)


def safe_corr(a: np.ndarray, b: np.ndarray):
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray):
    corrs = [safe_corr(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]
    maes = np.mean(np.abs(y_true - y_pred), axis=0)

    valid = np.array([not np.isnan(c) for c in corrs], dtype=bool)
    mean_corr = float(np.mean(np.array(corrs)[valid])) if valid.any() else np.nan

    return {
        "corrs": corrs,
        "mean_corr": mean_corr,
        "maes": maes,
    }


alphas = [3.0, 10.0, 30.0, 100.0]
results = []

for alpha in alphas:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha, fit_intercept=True))
    ])

    t0 = time.time()
    pipe.fit(X_train, Y_train, ridge__sample_weight=W_train)
    y_val_pred = pipe.predict(X_val).astype(np.float32)

    # controller-like clipping
    y_val_pred[:, :4] = np.clip(y_val_pred[:, :4], -1.0, 1.0)
    y_val_pred[:, 4:10] = np.clip(y_val_pred[:, 4:10], 0.0, 1.0)

    metrics = evaluate_predictions(Y_val, y_val_pred)
    dt = time.time() - t0

    results.append((alpha, metrics["mean_corr"], metrics["corrs"], metrics["maes"]))
    print(f"alpha={alpha:6.1f} | val_mean_corr={metrics['mean_corr']:.4f} | time={dt:.1f}s")

best_alpha = max(results, key=lambda x: x[1])[0]
print("\nBest alpha:", best_alpha)

for alpha, mean_corr, corrs, maes in results:
    print(f"\nalpha={alpha}")
    for i, c in enumerate(corrs):
        print(f"  gt_{i}: corr={c:.4f} mae={maes[i]:.4f}")

Train files:
 - smoke_Ani_seed1.h5
 - smoke_Ani_seed2.h5
 - smoke_Ani_seed3.h5
 - smoke_Ani_seed4.h5
Val file:
 - smoke_Ani_seed5.h5
X_train: (62047, 1280)
Y_train: (62047, 10)
W_train: (62047,)
X_val: (13394, 1280)
Y_val: (13394, 10)
W_val: (13394,)
alpha=   3.0 | val_mean_corr=0.6865 | time=1.9s
alpha=  10.0 | val_mean_corr=0.6864 | time=1.4s
alpha=  30.0 | val_mean_corr=0.6864 | time=1.4s
alpha= 100.0 | val_mean_corr=0.6860 | time=1.5s

Best alpha: 3.0

alpha=3.0
  gt_0: corr=0.7687 mae=0.0654
  gt_1: corr=0.6984 mae=0.0673
  gt_2: corr=0.7086 mae=0.0709
  gt_3: corr=0.6703 mae=0.0705
  gt_4: corr=0.6062 mae=0.0332
  gt_5: corr=0.7468 mae=0.0300
  gt_6: corr=0.5533 mae=0.0269
  gt_7: corr=0.5828 mae=0.0290
  gt_8: corr=0.6651 mae=0.0392
  gt_9: corr=0.8643 mae=0.0383

alpha=10.0
  gt_0: corr=0.7688 mae=0.0652
  gt_1: corr=0.6985 mae=0.0671
  gt_2: corr=0.7087 mae=0.0707
  gt_3: corr=0.6704 mae=0.0703
  gt_4: corr=0.6062 mae=0.0331
  gt_5: corr=0.7466 mae=0.0299
  gt_6: corr=0.5532 m

In [ ]:
# Free previous split arrays if needed
del X_train, Y_train, W_train, X_val, Y_val, W_val
gc.collect()

all_indices = list(range(len(binned_files)))
X_all, Y_all, W_all = build_split(all_indices)

print("X_all:", X_all.shape)
print("Y_all:", Y_all.shape)
print("W_all:", W_all.shape)

final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=best_alpha, fit_intercept=True))
])

t0 = time.time()
final_pipe.fit(X_all, Y_all, ridge__sample_weight=W_all)
print(f"Final fit finished in {time.time() - t0:.1f}s")

# quick training-set sanity check
y_all_pred = final_pipe.predict(X_all).astype(np.float32)
y_all_pred[:, :4] = np.clip(y_all_pred[:, :4], -1.0, 1.0)
y_all_pred[:, 4:10] = np.clip(y_all_pred[:, 4:10], 0.0, 1.0)

train_metrics = evaluate_predictions(Y_all, y_all_pred)
print("Train mean corr:", train_metrics["mean_corr"])
for i, c in enumerate(train_metrics["corrs"]):
    print(f"  gt_{i}: corr={c:.4f} mae={train_metrics['maes'][i]:.4f}")

X_all: (75441, 1280)
Y_all: (75441, 10)
W_all: (75441,)
Final fit finished in 1.6s
Train mean corr: 0.7209963053270985
  gt_0: corr=0.7208 mae=0.0612
  gt_1: corr=0.7198 mae=0.0641
  gt_2: corr=0.7379 mae=0.0674
  gt_3: corr=0.6588 mae=0.0679
  gt_4: corr=0.6675 mae=0.0312
  gt_5: corr=0.7048 mae=0.0294
  gt_6: corr=0.6722 mae=0.0271
  gt_7: corr=0.6780 mae=0.0284
  gt_8: corr=0.8352 mae=0.0386
  gt_9: corr=0.8149 mae=0.0348


In [ ]:
skl_onnx_path = os.path.join(OUT_DIR, "model_sklearn.onnx")

initial_type = [("input", FloatTensorType([None, CONTEXT_BINS * ENCODED_CHANNELS]))]
onnx_model = to_onnx(final_pipe, initial_types=initial_type, target_opset=17)

with open(skl_onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Saved sklearn ONNX to:", skl_onnx_path)

Saved sklearn ONNX to: /content/decoder_outputs_ridge/model_sklearn.onnx


In [ ]:
scaler = final_pipe.named_steps["scaler"]
ridge = final_pipe.named_steps["ridge"]

x_mean = scaler.mean_.astype(np.float32)
x_scale = scaler.scale_.astype(np.float32)
x_scale = np.where(x_scale < 1e-8, 1.0, x_scale).astype(np.float32)

W = ridge.coef_.astype(np.float32)        # shape: (10, 1280)
b = ridge.intercept_.astype(np.float32)   # shape: (10,)

print("x_mean:", x_mean.shape)
print("x_scale:", x_scale.shape)
print("W:", W.shape)
print("b:", b.shape)


class RidgeDecoderWrapper(nn.Module):
    def __init__(self, x_mean, x_scale, W, b):
        super().__init__()
        self.register_buffer("x_mean", torch.tensor(x_mean, dtype=torch.float32).view(1, -1))
        self.register_buffer("x_scale", torch.tensor(x_scale, dtype=torch.float32).view(1, -1))
        self.register_buffer("W", torch.tensor(W, dtype=torch.float32))   # [10, 1280]
        self.register_buffer("b", torch.tensor(b, dtype=torch.float32).view(1, -1))

    def forward(self, x):
        # x: [batch, 1280]
        x = (x - self.x_mean) / self.x_scale
        y10 = x @ self.W.T + self.b

        # clip to controller-like ranges
        y_signed = torch.clamp(y10[:, :4], -1.0, 1.0)
        y_pos = torch.clamp(y10[:, 4:10], 0.0, 1.0)
        y10 = torch.cat([y_signed, y_pos], dim=1)

        # convert back to original training value range
        y10[:, :4] = y10[:, :4] * 32767.0
        y10[:, 4:10] = y10[:, 4:10] * 32767.0

        zeros = torch.zeros((x.shape[0], 2), dtype=x.dtype, device=x.device)
        y12 = torch.cat([y10, zeros], dim=1)
        return y12


wrapper = RidgeDecoderWrapper(x_mean, x_scale, W, b).eval()

x_mean: (1280,)
x_scale: (1280,)
W: (10, 1280)
b: (10,)


In [ ]:
final_onnx_path = os.path.join(OUT_DIR, "model.onnx")
dummy = torch.randn(1, CONTEXT_BINS * ENCODED_CHANNELS, dtype=torch.float32)

torch.onnx.export(
    wrapper,
    dummy,
    final_onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch"},
        "output": {0: "batch"},
    },
    opset_version=17,
)

print("Saved final ONNX to:", final_onnx_path)

/tmp/ipykernel_6130/3961168891.py:4: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0412 14:59:44.264000 6130 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `RidgeDecoderWrapper()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `RidgeDecoderWrapper()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 9 of general pattern rewrite rules.
Applied 2 of general pattern rewrite rules.
Saved final ONNX to: /content/decoder_outputs_ridge/model.onnx


In [ ]:
sess = ort.InferenceSession(final_onnx_path, providers=["CPUExecutionProvider"])

x_test = np.random.randn(4, CONTEXT_BINS * ENCODED_CHANNELS).astype(np.float32)
onnx_out = sess.run(None, {"input": x_test})[0]

with torch.no_grad():
    torch_out = wrapper(torch.from_numpy(x_test)).numpy()

print("onnx_out shape:", onnx_out.shape)
print("torch_out shape:", torch_out.shape)
print("max abs diff:", np.max(np.abs(onnx_out - torch_out)))

onnx_out shape: (4, 12)
torch_out shape: (4, 12)
max abs diff: 0.0009765625


In [ ]:
meta = {
    "data_dir": DATA_DIR,
    "files": [os.path.basename(f) for f in FILES],
    "dataset_key": DATASET_KEY,
    "total_channels": TOTAL_CHANNELS,
    "encoded_channels": ENCODED_CHANNELS,
    "ground_truth_channels": GT_CHANNELS,
    "predicted_channels": list(range(10)),
    "constant_zero_channels": [10, 11],
    "sample_rate_hz": FS,
    "bin_size_raw_frames": BIN_SIZE,
    "bin_duration_ms": 1000.0 * BIN_SIZE / FS,
    "context_bins": CONTEXT_BINS,
    "context_duration_ms": 1000.0 * BIN_SIZE * CONTEXT_BINS / FS,
    "onnx_input_shape": [1, CONTEXT_BINS * ENCODED_CHANNELS],
    "onnx_output_shape": [1, GT_CHANNELS],
    "best_alpha": float(best_alpha),
    "notes": (
        "Windowed ridge decoder. Input is 20 bins x 64 channels flattened to 1280. "
        "Outputs gt_0..gt_9 as clipped controller-like values in the original scale; "
        "gt_10 and gt_11 are hard-set to zero."
    ),
    "synapse_app_parameters": {
        "model_name": "model",
        "input_channels": 64,
        "frames_per_step": 320
    }
}

meta_path = os.path.join(OUT_DIR, "decoder_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta to:", meta_path)
print("Main file for deployment:", final_onnx_path)

Saved meta to: /content/decoder_outputs_ridge/decoder_meta.json
Main file for deployment: /content/decoder_outputs_ridge/model.onnx


In [ ]:
import json

with open("/content/decoder_outputs_ridge/decoder_meta.json", "r") as f:
    meta = json.load(f)

fs = meta["sample_rate_hz"]                          # 32000
frames_per_step = meta["synapse_app_parameters"]["frames_per_step"]  # 320
steps_per_sec = fs / frames_per_step

n_total_outputs = meta["ground_truth_channels"]      # 12
n_pred_outputs = len(meta["predicted_channels"])     # 10

for bits_per_output in [8, 16]:
    bitrate_total = steps_per_sec * n_total_outputs * bits_per_output
    bitrate_pred = steps_per_sec * n_pred_outputs * bits_per_output
    print(f"{bits_per_output}-bit, 12 outputs: {bitrate_total:.1f} bps ({bitrate_total/1000:.2f} kbps)")
    print(f"{bits_per_output}-bit, 10 learned outputs: {bitrate_pred:.1f} bps ({bitrate_pred/1000:.2f} kbps)")

8-bit, 12 outputs: 9600.0 bps (9.60 kbps)
8-bit, 10 learned outputs: 8000.0 bps (8.00 kbps)
16-bit, 12 outputs: 19200.0 bps (19.20 kbps)
16-bit, 10 learned outputs: 16000.0 bps (16.00 kbps)


In [ ]:
import numpy as np

def quantize_outputs(y, bits=8):
    """
    y: [T, 12] or [T, 10] decoder outputs in original controller scale.
       Expected ranges from notebook:
       - channels 0:4 are signed, roughly [-32767, 32767]
       - channels 4:10 are nonnegative, roughly [0, 32767]
    Returns integer-coded array.
    """
    y = np.asarray(y).copy()

    if y.shape[1] >= 4:
        y[:, :4] = np.clip(y[:, :4], -32767, 32767)
    if y.shape[1] > 4:
        y[:, 4:] = np.clip(y[:, 4:], 0, 32767)

    levels = 2 ** bits

    q = np.zeros_like(y, dtype=np.int32)

    # signed channels -> map [-32767,32767] to [0, levels-1]
    q[:, :4] = np.round((y[:, :4] + 32767) * (levels - 1) / (2 * 32767)).astype(np.int32)

    # nonnegative channels -> map [0,32767] to [0, levels-1]
    if y.shape[1] > 4:
        q[:, 4:] = np.round(y[:, 4:] * (levels - 1) / 32767).astype(np.int32)

    q = np.clip(q, 0, levels - 1)
    return q


def entropy_bits(x, levels=None):
    x = np.asarray(x).astype(np.int64).ravel()
    if levels is None:
        levels = int(x.max()) + 1
    counts = np.bincount(x, minlength=levels).astype(np.float64)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())


def estimate_entropy_bitrate(y_pred, steps_per_sec=100, bits=8):
    """
    y_pred: [T, C] predictions in original controller scale
    Returns total entropy rate and per-channel entropies.
    """
    q = quantize_outputs(y_pred, bits=bits)
    levels = 2 ** bits

    per_channel_entropy = [entropy_bits(q[:, c], levels=levels) for c in range(q.shape[1])]
    bits_per_step = float(np.sum(per_channel_entropy))
    bits_per_sec = bits_per_step * steps_per_sec

    return {
        "bits_per_step": bits_per_step,
        "bits_per_sec": bits_per_sec,
        "kbps": bits_per_sec / 1000.0,
        "per_channel_entropy_bits": per_channel_entropy,
    }

# Example:
# y_pred should be your decoder outputs in original scale, shape [T, 10] or [T, 12]
# result = estimate_entropy_bitrate(y_pred, steps_per_sec=100, bits=8)
# print(result)

In [ ]:
import onnx

model_path = "/content/decoder_outputs_ridge/model.onnx"
patched_path = "/content/decoder_outputs_ridge/model_ir9.onnx"

m = onnx.load(model_path)

print("Before:")
print("  ir_version =", m.ir_version)
print("  opsets =", [(op.domain, op.version) for op in m.opset_import])

# Device ORT 1.16 supports IR version 9
m.ir_version = 9

onnx.save(m, patched_path)

m2 = onnx.load(patched_path)
print("After:")
print("  ir_version =", m2.ir_version)
print("  opsets =", [(op.domain, op.version) for op in m2.opset_import])

Before:
  ir_version = 10
  opsets = [('', 17)]
After:
  ir_version = 9
  opsets = [('', 17)]
